# WildfireBenchBaselines — Colab training

Train a WildfireSpreadTS baseline (ResNet18 U-Net by default) and log **AP, F1, IoU, Precision, Recall** to Weights & Biases.

**Before you start**
1. Runtime → Change runtime type → **GPU** (T4 is fine).
2. Make sure your HDF5 dataset is in Google Drive (a folder containing year subfolders `2018/`, `2019/`, `2020/`, `2021/`, each with `*.hdf5` fires).
3. Run the cells top to bottom. You only need to edit **`DATA_DIR`** in the Drive cell.

Metrics come from the model's native `test_step` (torchmetrics) plus the periodic `UnifiedEvalCallback` (every 5 epochs), logged to W&B under `test_*` and `eval/*`.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Clone the repo

In [ ]:
import os

REPO_URL = "https://github.com/mpannozzo/WildfireBenchBaselines.git"
REPO_DIR = "/content/WildfireBenchBaselines"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

## 3. Install dependencies

Uses Colab's preinstalled CUDA build of PyTorch (don't reinstall torch). Installs the lightweight deps plus a compatible PyTorch Lightning.

In [ ]:
!pip install -q -r requirements-colab.txt
!pip install -q "pytorch-lightning>=2.1,<2.4"

import torch, pytorch_lightning as pl
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
print("lightning:", pl.__version__)

## 4. Mount Google Drive and point to your dataset

**Edit `DATA_DIR`** to the folder that holds your HDF5 year subfolders.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Folder containing 2018/ 2019/ 2020/ 2021/ subfolders of *.hdf5 files
DATA_DIR = "/content/drive/MyDrive/newHDF5Data"

import os
assert os.path.isdir(DATA_DIR), f"DATA_DIR not found: {DATA_DIR}"
print("DATA_DIR =", DATA_DIR)
print("Top-level contents:", sorted(os.listdir(DATA_DIR))[:20])

## 5. (Optional but recommended) Build the HDF5 index

Writes `dataset_index.json` to your data folder so future sessions skip scanning every fire on Drive at startup. Run once per dataset; safe to re-run (it resumes). Skip if you've already built it.

In [ ]:
!python {REPO_DIR}/src/preprocess/BuildHDF5Index.py --data_dir "{DATA_DIR}"

## 6. Log in to Weights & Biases

Paste your API key from https://wandb.ai/authorize when prompted. (To run without an account, replace this cell with `%env WANDB_MODE=offline`.)

In [ ]:
import wandb
wandb.login()

## 7. Train (+ test)

Defaults: ResNet18 U-Net, 1 day of observations, all features, fold 0 (test year = 2021).

Uses `cfgs/trainer_colab.yaml`, which writes checkpoints to **`/content/drive/MyDrive/wildfire_runs/checkpoints`** on your Drive (with a `last.ckpt`) so they survive Colab disconnects and can be resumed.

- `EPOCHS=50` is a Colab-sized budget (the paper uses ~200, which won't finish in one session). Raise it once you can resume across sessions.
- The first epoch is slow because each fire's HDF5 is copied from Drive to local disk; later epochs run on the warm cache.
- If you hit CUDA out-of-memory, lower `BATCH_SIZE` (e.g. 16).
- `do_test=True` runs the full test set at the end and logs `test_AP`, `test_f1`, `test_iou`, `test_precision`, `test_recall` to W&B.

In [ ]:
EPOCHS = 50
BATCH_SIZE = 32

%cd {REPO_DIR}/src
!python train.py \
  --config=../cfgs/unet/res18_monotemporal.yaml \
  --trainer=../cfgs/trainer_colab.yaml \
  --data=../cfgs/data_monotemporal_full_features.yaml \
  --seed_everything=0 \
  --trainer.max_epochs={EPOCHS} \
  --trainer.precision=16-mixed \
  --data.data_dir="{DATA_DIR}" \
  --data.batch_size={BATCH_SIZE} \
  --data.num_workers=2 \
  --do_test=True

## 8. Resume after a disconnect

If Colab dropped mid-run, re-run cells 1-4 (clone, install, mount Drive), then run this cell. It picks up `last.ckpt` from Drive and continues training. Bump `EPOCHS` higher than where you left off.

In [ ]:
import os

CKPT = "/content/drive/MyDrive/wildfire_runs/checkpoints/last.ckpt"
assert os.path.isfile(CKPT), f"No checkpoint found at {CKPT}"

EPOCHS = 50
BATCH_SIZE = 32

%cd {REPO_DIR}/src
!python train.py \
  --config=../cfgs/unet/res18_monotemporal.yaml \
  --trainer=../cfgs/trainer_colab.yaml \
  --data=../cfgs/data_monotemporal_full_features.yaml \
  --seed_everything=0 \
  --trainer.max_epochs={EPOCHS} \
  --trainer.precision=16-mixed \
  --data.data_dir="{DATA_DIR}" \
  --data.batch_size={BATCH_SIZE} \
  --data.num_workers=2 \
  --ckpt_path="{CKPT}" \
  --do_test=True

## Notes & variations

- **Other models:** swap the `--config` file, e.g. `../cfgs/convlstm/full_run.yaml`, `../cfgs/UTAE/all_features.yaml`, or `../cfgs/LogisticRegression/full_run.yaml`.
- **Different fold / test year:** add `--data.data_fold_id=N` (0–11). Fold 0 uses test year 2021.
- **Quick smoke test** (verify the pipeline end-to-end fast): add `--trainer.limit_train_batches=2 --trainer.limit_val_batches=2 --trainer.limit_test_batches=2 --trainer.max_epochs=1`.
- **Test only from a checkpoint:** add `--do_train=False --ckpt_path=/path/to/ckpt.ckpt`.
- Checkpoints/logs are written under `src/lightning_logs/` and (with `log_model: true`) uploaded to W&B.